In [1]:
import os
!pip install python-dotenv
from dotenv import load_dotenv
import pandas as pd
from datetime import datetime
!pip install wordfilter
from wordfilter import Wordfilter
import streamlit as st
import openai

load_dotenv("env")

os.environ['AUTOGEN_USE_DOCKER'] = '0'

In [2]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager


llm_config = {
    "model": "deepseek-chat",
    "api_key": os.getenv("DEEPSEEK_API_KEY"),
    "base_url": os.getenv("DEEPSEEK_BASE_URL")
}

In [3]:
wordfilter = Wordfilter()

def contains_sensitive_words(prompt):
    return wordfilter.blacklisted(prompt)

In [4]:
# Define the agents
prompt_generator_agent = AssistantAgent(
    name="Prompt_Generator_Agent",
    system_message="""You are an agent that generates the initial prompt based on a broad description of the user's task.
Do:
-Provide a clear and concise initial prompt.
-Focus on capturing the essence of the task.
Don't:
-Overcomplicate the prompt with unnecessary details.
Example:
Input: "What is diabetes?"
Good Output: "Explain the key aspects of diabetes."
Bad Output: "I don't know, just say something about it."
""",
    llm_config=llm_config  
)

clarity_agent = AssistantAgent(
    name="Clarity_Agent",
    system_message="""You are an agent that enhances the clarity and understandability of the prompt.
Do:
- Rephrase the prompt to be easily understood.
- Remove ambiguity.
Don't:
- Change the intended meaning.
Example:
Input: "Explain diabetes."
Good Output: "Describe the causes, symptoms, and treatments of diabetes in simple terms."
Bad Output: "Explain diabetes in a convoluted manner with excessive jargon."
""",
    llm_config=llm_config  
)

relevance_agent = AssistantAgent(
    name="Relevance_Agent",
    system_message="""You are an agent that ensures the prompt remains focused on the user's original intent.
Do:
- Keep the response on topic.
- Remove extraneous or irrelevant information.
Don't:
- Introduce unrelated details.
Example:
Input: "Discuss diabetes in the context of modern medicine."
Good Output: "Explain diabetes with a focus on current treatment methods and research."
Bad Output: "Discuss diabetes and unrelated historical facts."
""",
    llm_config=llm_config  
)

precision_agent = AssistantAgent(
    name="Precision_Agent",
    system_message="""You are an agent that increases the precision of the prompt.
Do:
- Ask for or include specific details where necessary.
- Narrow down the scope to be more specific.
Don't:
- Provide vague or overly general responses.
Example:
Input: "Explain diabetes."
Good Output: "Detail the symptoms and treatment options for type 2 diabetes."
Bad Output: "Explain diabetes in a general way without specifics."
""",
    llm_config=llm_config  
)

creativity_agent = AssistantAgent(
    name="Creativity_Agent",
    system_message="""You are an agent that encourages more imaginative and innovative outputs based on what would delight the end user.
Do:
- Introduce creative angles or analogies when appropriate.
- Enhance the prompt with creative suggestions.
Don't:
- Overcomplicate or confuse the prompt.
Example:
Input: "Explain diabetes."
Good Output: "Describe diabetes using a creative analogy, such as comparing blood sugar regulation to a thermostat."
Bad Output: "Use random creative words that do not contribute to understanding."
""",
    llm_config=llm_config  
)

completeness_agent = AssistantAgent(
    name="Completeness_Agent",
    system_message="""You are the Refinement Agent. Your role is to synthesize and refine the outputs from the previous agents into a final, concise prompt.
Do:
- Combine key information from each agent into a coherent, actionable prompt.
- Ensure that the final prompt captures all essential details.
Don't:
- Simply concatenate outputs without integration.
- Include irrelevant or redundant information.
Example:
If the responses are:
Prompt_Generator_Agent: "Describe diabetes."
Clarity_Agent: "Diabetes is a chronic condition affecting blood sugar regulation."
Relevance_Agent: "Focus on type 2 diabetes if applicable."
Precision_Agent: "Specify symptoms and treatment options."
Creativity_Agent: "Consider using a creative analogy."
A good final refined prompt could be:
"Provide a detailed explanation of type 2 diabetes, including its symptoms, treatment options, and an illustrative creative analogy."
""",
    llm_config=llm_config  
)

# ------------------------
# Define the QA Agent as the Improved Prompt Generator
# ------------------------

qa_agent = AssistantAgent(
    name="QA_Agent",
    system_message="""You are the QA Agent.
Your task is to take the user's original prompt along with the refined output from the other agents and generate an improved initial prompt.
This improved prompt should more accurately capture the user's intended request and be more likely to achieve a valuable output.
Do:
- Consider both the original prompt and the additional details provided by the other agents.
- Generate a revised version that is clear, focused, and actionable.
Don't:
- Simply echo the original prompt or include irrelevant details.
Example:
User's initial prompt: "what is diabitas"
Other agents' outputs refine and clarify the intent to discuss diabetes.
Improved Prompt: "Provide a detailed explanation of diabetes, including its causes, symptoms, and treatment options."
""",
    llm_config=llm_config  
)


# Define the user proxy agent
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",  # Automatically respond without human input
    max_consecutive_auto_reply=10,
    llm_config=llm_config  
)

# Create a group chat with all agents
groupchat = GroupChat(
    agents=[user_proxy, prompt_generator_agent, clarity_agent, relevance_agent, precision_agent, creativity_agent, completeness_agent],
    messages=[],
    max_round=10
)

In [5]:
# Create a group chat manager
manager = GroupChatManager(groupchat=groupchat)

In [6]:
# Initialize a DataFrame to store prompt history
history_file = "prompt_history_general.xlsx"
if not os.path.exists(history_file):
    df = pd.DataFrame(columns=["Timestamp", "Input Prompt", "Prompt_Generator_Agent", "Clarity_Agent", "Relevance_Agent", "Precision_Agent", "Creativity_Agent", "Final Refined Prompt", "Improved Prompt"])
    df.to_excel(history_file, index=False)

In [7]:
# Function to distribute the prompt to all agents simultaneously
def run_parallel_agents(prompt):
    if contains_sensitive_words(prompt):
        print("The input contains sensitive words. Please rephrase your question.")
        return None
    
    responses = {}
    messages = [{"role": "user", "content": prompt}]
    
    for agent in [prompt_generator_agent, clarity_agent, relevance_agent, precision_agent, creativity_agent]:
        responses[agent.name] = agent.generate_reply(messages)
    
    # Display each agent's output
    for name, output in responses.items():
        print(f"{name} Output: {output}\n")
    
    # Aggregate responses into a final refinement request
    combined_input = "\n".join([f"{name}: {resp}" for name, resp in responses.items()])
    final_messages = [{"role": "user", "content": combined_input}]
    refined_prompt = completeness_agent.generate_reply(final_messages)
    
    print("Final Refined Prompt:", refined_prompt)
    
   # Save results to Excel with timestamp
    df = pd.read_excel(history_file)
    new_entry = {"Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "Input Prompt": prompt, "Final Refined Prompt": refined_prompt}
    for name in responses:
            new_entry[name] = responses[name]
    df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
    df.to_excel(history_file, index=False)
    
    return refined_prompt

def run_qa_agent(initial_prompt, refined_prompt):
    # The QA agent takes both the user's original prompt and the refined prompt from the other agents
    qa_messages = [
        {"role": "user", "content": f"User's initial prompt: {initial_prompt}"},
        {"role": "user", "content": f"Refined details: {refined_prompt}"}
    ]
    improved_prompt = qa_agent.generate_reply(qa_messages)
    print("Improved Initial Prompt:", improved_prompt)
    return improved_prompt